# Feature Engineering for Electricity Demand Forecasting

**Purpose:** Create advanced features from existing IESO demand + weather data to improve model performance.

**Current baseline:** Linear Regression with basic features (RMSE: 2,215.79 MW, R² = 0.151)

**Feature categories to engineer:**

1. **Lag Features** - Historical demand patterns
   - Demand at t-1 (previous hour)
   - Demand at t-24 (same hour yesterday)
   - Demand at t-168 (same hour last week)

2. **Rolling Statistics** - Trend indicators
   - 24-hour rolling average (daily trend)
   - 168-hour rolling average (weekly trend)

3. **Degree Days** - Heating/Cooling demand drivers
   - Heating Degree Days (HDD): when temp < 18°C
   - Cooling Degree Days (CDD): when temp > 18°C

4. **Temperature Interactions** - Non-linear HVAC response
   - Temperature × Hour (time-of-day effect)
   - Temperature² (quadratic relationship)

5. **Cyclical Encoding** - Periodic patterns
   - Hour (sin/cos encoding for 24-hour cycle)
   - Month (sin/cos encoding for annual cycle)

6. **Temporal Indicators** - Calendar effects
   - IsWeekend (binary flag)
   - DayOfWeek (0-6)
   - Month, Quarter, Season

**Expected improvement:** RMSE reduction of 20-40% through engineered features

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)

# Load the existing merged IESO + Weather data
data_path = "../../02_Datasets/processed/final_merged_ieso_weather.csv"
df = pd.read_csv(data_path, parse_dates=['DateTime'])

print("✓ Data loaded successfully")
print(f"\nOriginal dataset shape: {df.shape}")
print(f"Date range: {df['DateTime'].min()} to {df['DateTime'].max()}")
print(f"\nCurrent columns ({len(df.columns)}):")
print(df.columns.tolist())
print(f"\nFirst 3 rows:")
df.head(3)

✓ Data loaded successfully

Original dataset shape: (109056, 17)
Date range: 2013-06-01 00:00:00 to 2025-11-09 00:00:00

Current columns (17):
['DateTime', 'Date', 'Hour', 'Year', 'Month', 'DayOfWeek', 'DayOfYear', 'IsWeekend', 'Market Demand', 'Ontario Demand', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Wind Dir (10s deg)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)']

First 3 rows:


,DateTime,Date,Hour,Year,Month,DayOfWeek,DayOfYear,IsWeekend,Market Demand,Ontario Demand,Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Dir (10s deg),Wind Spd (km/h),Visibility (km),Stn Press (kPa)
0,2013-06-01 00:00:00,2013-06-01,1,2013,6,5,152,1,15908,13731,17.8,16.0,89.0,0.0,21.0,24.1,98.66
1,2013-06-01 01:00:00,2013-06-01,2,2013,6,5,152,1,15126,13143,17.8,16.0,89.0,0.0,21.0,24.1,98.66
2,2013-06-01 02:00:00,2013-06-01,3,2013,6,5,152,1,14652,12778,17.8,16.0,89.0,0.0,21.0,24.1,98.66


In [2]:
print("Creating lag features...")

# Sort by DateTime to ensure proper ordering
df = df.sort_values('DateTime').reset_index(drop=True)

# Lag features - previous demand values
df['Demand_Lag_1h'] = df['Ontario Demand'].shift(1)      # 1 hour ago
df['Demand_Lag_24h'] = df['Ontario Demand'].shift(24)    # Same hour yesterday
df['Demand_Lag_168h'] = df['Ontario Demand'].shift(168)  # Same hour last week

print(f"✓ Created 3 lag features")
print(f"\nNew shape: {df.shape}")

# Show example of lag features
print(f"\nExample of lag features (rows 170-172):")
print(df[['DateTime', 'Ontario Demand', 'Demand_Lag_1h', 'Demand_Lag_24h', 'Demand_Lag_168h']].iloc[170:173])

Creating lag features...
✓ Created 3 lag features

New shape: (109056, 20)

Example of lag features (rows 170-172):
               DateTime  Ontario Demand  Demand_Lag_1h  Demand_Lag_24h  \
170 2013-06-08 02:00:00           11690        11996.0         12133.0   
171 2013-06-08 03:00:00           11643        11690.0         12282.0   
172 2013-06-08 04:00:00           11741        11643.0         12687.0   

     Demand_Lag_168h  
170          12778.0  
171          12509.0  
172          12574.0  


In [3]:
print("Creating rolling average features...")

# Rolling averages - trend indicators
df['Demand_RollingAvg_24h'] = df['Ontario Demand'].rolling(window=24, min_periods=1).mean()
df['Demand_RollingAvg_168h'] = df['Ontario Demand'].rolling(window=168, min_periods=1).mean()

print(f"✓ Created 2 rolling average features")
print(f"\nNew shape: {df.shape}")

# Show example
print(f"\nExample of rolling averages (rows 170-172):")
print(df[['DateTime', 'Ontario Demand', 'Demand_RollingAvg_24h', 'Demand_RollingAvg_168h']].iloc[170:173])

Creating rolling average features...
✓ Created 2 rolling average features

New shape: (109056, 22)

Example of rolling averages (rows 170-172):
               DateTime  Ontario Demand  Demand_RollingAvg_24h  \
170 2013-06-08 02:00:00           11690           14779.000000   
171 2013-06-08 03:00:00           11643           14752.375000   
172 2013-06-08 04:00:00           11741           14712.958333   

     Demand_RollingAvg_168h  
170            14789.220238  
171            14784.065476  
172            14779.107143  


In [4]:
print("Creating degree day features...")

# Base temperature for degree days (18°C is standard for Ontario)
base_temp = 18

# Heating Degree Days (HDD) - when temperature < 18°C
df['HDD'] = np.maximum(base_temp - df['Temp (°C)'], 0)

# Cooling Degree Days (CDD) - when temperature > 18°C  
df['CDD'] = np.maximum(df['Temp (°C)'] - base_temp, 0)

print(f"✓ Created 2 degree day features (HDD, CDD)")
print(f"\nNew shape: {df.shape}")

# Show examples across different temperatures
print(f"\nExamples across temperature range:")
sample_indices = [100, 5000, 50000, 80000, 100000]
print(df[['DateTime', 'Temp (°C)', 'HDD', 'CDD']].iloc[sample_indices])

Creating degree day features...
✓ Created 2 degree day features (HDD, CDD)

New shape: (109056, 24)

Examples across temperature range:
                  DateTime  Temp (°C)   HDD  CDD
100    2013-06-05 04:00:00       17.8   0.2  0.0
5000   2013-12-26 08:00:00       -4.1  22.1  0.0
50000  2019-02-13 08:00:00       -2.5  20.5  0.0
80000  2022-07-17 08:00:00       24.4   0.0  6.4
100000 2024-10-27 16:00:00       11.2   6.8  0.0


In [5]:
print("Creating temperature interaction features...")

# Temperature × Hour interaction (HVAC demand varies by time of day)
df['Temp_x_Hour'] = df['Temp (°C)'] * df['Hour']

# Temperature squared (quadratic relationship for HVAC)
df['Temp_Squared'] = df['Temp (°C)'] ** 2

print(f"✓ Created 2 temperature interaction features")
print(f"\nNew shape: {df.shape}")

# Show examples
print(f"\nExamples:")
print(df[['DateTime', 'Hour', 'Temp (°C)', 'Temp_x_Hour', 'Temp_Squared']].iloc[170:175])

Creating temperature interaction features...
✓ Created 2 temperature interaction features

New shape: (109056, 26)

Examples:
               DateTime  Hour  Temp (°C)  Temp_x_Hour  Temp_Squared
170 2013-06-08 02:00:00     3       17.8         53.4        316.84
171 2013-06-08 03:00:00     4       17.8         71.2        316.84
172 2013-06-08 04:00:00     5       17.8         89.0        316.84
173 2013-06-08 05:00:00     6       17.8        106.8        316.84
174 2013-06-08 06:00:00     7       17.8        124.6        316.84


In [6]:
print("Creating cyclical encoding features...")

# Hour encoding (24-hour cycle)
# Sin/cos transforms make hour 23 close to hour 0 (circular pattern)
df['Hour_Sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_Cos'] = np.cos(2 * np.pi * df['Hour'] / 24)

# Month encoding (12-month cycle)
# Sin/cos transforms make December close to January (circular pattern)
df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)

print(f"✓ Created 4 cyclical encoding features")
print(f"\nNew shape: {df.shape}")

# Show how cyclical encoding works
print(f"\nCyclical encoding examples:")
sample_hours = [0, 6, 12, 18, 23]
print(df[['Hour', 'Hour_Sin', 'Hour_Cos']].iloc[sample_hours])

Creating cyclical encoding features...
✓ Created 4 cyclical encoding features

New shape: (109056, 30)

Cyclical encoding examples:
    Hour      Hour_Sin  Hour_Cos
0      1  2.588190e-01  0.965926
6      7  9.659258e-01 -0.258819
12    13 -2.588190e-01 -0.965926
18    19 -9.659258e-01  0.258819
23    24 -2.449294e-16  1.000000


In [7]:
# Save the engineered dataset
output_path = "../../02_Datasets/processed/ieso_weather_engineered_features.csv"
df.to_csv(output_path, index=False)

print(f"✓ Engineered dataset saved to: {output_path}")
print(f"\n{'='*70}")
print(f"FEATURE ENGINEERING COMPLETE!")
print(f"{'='*70}")

print(f"\nDataset transformation:")
print(f"  Original columns: 17")
print(f"  New columns: {len(df.columns)}")
print(f"  Features added: {len(df.columns) - 17}")

print(f"\nNew features created:")
print(f"  Lag features (3): Demand_Lag_1h, Demand_Lag_24h, Demand_Lag_168h")
print(f"  Rolling averages (2): Demand_RollingAvg_24h, Demand_RollingAvg_168h")
print(f"  Degree days (2): HDD, CDD")
print(f"  Temperature interactions (2): Temp_x_Hour, Temp_Squared")
print(f"  Cyclical encoding (4): Hour_Sin, Hour_Cos, Month_Sin, Month_Cos")

print(f"\nFinal shape: {df.shape}")
print(f"Date range: {df['DateTime'].min()} to {df['DateTime'].max()}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\nAll columns ({len(df.columns)}):")
print(df.columns.tolist())

✓ Engineered dataset saved to: ../../02_Datasets/processed/ieso_weather_engineered_features.csv

FEATURE ENGINEERING COMPLETE!

Dataset transformation:
  Original columns: 17
  New columns: 30
  Features added: 13

New features created:
  Lag features (3): Demand_Lag_1h, Demand_Lag_24h, Demand_Lag_168h
  Rolling averages (2): Demand_RollingAvg_24h, Demand_RollingAvg_168h
  Degree days (2): HDD, CDD
  Temperature interactions (2): Temp_x_Hour, Temp_Squared
  Cyclical encoding (4): Hour_Sin, Hour_Cos, Month_Sin, Month_Cos

Final shape: (109056, 30)
Date range: 2013-06-01 00:00:00 to 2025-11-09 00:00:00
Missing values: 193
Memory usage: 30.27 MB

All columns (30):
['DateTime', 'Date', 'Hour', 'Year', 'Month', 'DayOfWeek', 'DayOfYear', 'IsWeekend', 'Market Demand', 'Ontario Demand', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Wind Dir (10s deg)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)', 'Demand_Lag_1h', 'Demand_Lag_24h', 'Demand_Lag_168h', 'Demand_RollingAvg_24h', '